In [ ]:
import polars as pl
from pathlib import Path
import subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed

In [ ]:
CWD = Path().resolve()
BASE = CWD

print(f"Folder where groups (data) folder should be: {BASE}")

In [ ]:
groups_map = {
    "plants_plt": ["solanaceae"],
    "plants_mit": ["solanaceae"],
}

In [ ]:
RUN_NAME = "clustalo_2026-09-30"
RE_RUN = False
WORKERS = 32

In [ ]:
def build_cmd(in_fasta, out_fasta, log_file):
    return [
        "clustalo",
        "-i",
        str(in_fasta),
        "-o",
        str(out_fasta),
        "--outfmt=fa",
        "-t",
        "DNA",
        "--full-iter",
        "--iter=3",
        "--output-order=input-order",
        "--threads=1",
        f"--log={log_file}",
        "--verbose",
        "--force",
    ]


def run_clustalo(gp, gp_dir, run_dir, logs_dir, force_run):
    in_fasta = gp_dir / f"gp_{gp}.fasta"
    out_fasta = run_dir / f"gp_{gp}_aligned.fasta"
    log_file = logs_dir / f"gp_{gp}_clustalo.log"

    if not in_fasta.exists():
        return gp, "missing input"
    if out_fasta.exists() and not force_run:
        return gp, "skipped"

    cmd = build_cmd(in_fasta, out_fasta, log_file)
    res = subprocess.run(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    if res.returncode != 0:
        (logs_dir / f"gp_{gp}_clustalo_error.log").write_text(res.stdout)
        return gp, f"error (exit {res.returncode})"
    return gp, "ok"

In [ ]:
for g, subgs in groups_map.items():
    for subg in subgs:
        subg_dir = BASE / g / subg
        print(f"Processing {g}/{subg}")

        gp_file = subg_dir / "gene_pairs_filtered.tsv"
        if not gp_file.exists():
            print(f"  no {gp_file.name}, skipping")
            continue

        gp_dir = subg_dir / "gene_pair_fastas"
        run_dir = subg_dir / "alignments" / RUN_NAME
        logs_dir = run_dir / "logs"
        logs_dir.mkdir(parents=True, exist_ok=True)

        # record exactly what was run, with placeholders for per-pair paths
        template = build_cmd(
            "<gp_dir>/gp_{gp}.fasta",
            "<run_dir>/gp_{gp}_aligned.fasta",
            "<logs_dir>/gp_{gp}_clustalo.log",
        )
        (run_dir / "params.txt").write_text(" ".join(template) + "\n")

        gp_df = pl.read_csv(gp_file, separator="\t")
        pairs = gp_df.sort(pl.col("igr_count") * pl.col("mean_len"), descending=True)[
            "gene_pair"
        ].to_list()

        with ThreadPoolExecutor(max_workers=WORKERS) as executor:
            tasks = [
                executor.submit(run_clustalo, gp, gp_dir, run_dir, logs_dir, RE_RUN)
                for gp in pairs
            ]

            for future in as_completed(tasks):
                gp, status = future.result()
                print(f"  {gp}: {status}")

plants_plt
    runtime: 55m27s
    pairs: 195

plants_mit
    runtime: 7m8s
    pairs: 32